# Module 8: Forecast Orchestration & Pipelines

This notebook demonstrates running the batch forecasting pipeline and inspecting its outputs.

You can run the pipeline either:
- from a terminal (recommended)
- from this notebook via `subprocess`


In [ ]:
from pathlib import Path
import json
import pandas as pd
import subprocess

print('Ready')


## Run the pipeline

If you already ran it from terminal, skip this cell.

Otherwise, this runs:

- `python pipelines/forecasting_pipeline.py --config config/pipeline.yaml`


In [ ]:
# Run the pipeline
cmd = ['python', '../pipelines/forecasting_pipeline.py', '--config', '../config/pipeline.yaml']
print('Running:', ' '.join(cmd))

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Pipeline failed with exit code {result.returncode}')


## Inspect outputs

We’ll load the most recent pipeline report and its forecast CSV.


In [ ]:
reports_dir = Path('../outputs/reports')
forecast_dir = Path('../outputs/forecasts')

report_files = sorted(reports_dir.glob('*_pipeline_report.json'), key=lambda p: p.stat().st_mtime, reverse=True)
if not report_files:
    raise FileNotFoundError('No pipeline reports found in outputs/reports')

latest_report = report_files[0]
print('Latest report:', latest_report.name)
report = json.loads(latest_report.read_text(encoding='utf-8'))
report


In [ ]:
forecast_path = Path(report['forecast']['output'])
if not forecast_path.is_absolute():
    # report stores repo-relative paths
    forecast_path = Path('..') / forecast_path

print('Forecast file:', forecast_path)
fc = pd.read_csv(forecast_path, parse_dates=['date'])
print('Shape:', fc.shape)
fc.head()


In [ ]:
# Quick sanity checks
print('Unique SKUs:', fc['sku_id'].nunique())
print('Forecast horizon days:', fc['date'].nunique())
print('Methods:', fc['method'].unique())

# Optional: category totals
if 'category' in fc.columns:
    cat_totals = fc.groupby(['date','category'])['y_pred'].sum().reset_index()
    cat_totals.head()
